# Member A -- Agreement Network Between Students

Nodes = students, edges = similarity of per-item z-scored opinion vectors.

**Data**: `Survey_Results_UC.csv`, read fresh and filtered with the
team's shared rule (drop any respondent with > 30 of 60 answers missing),
leaving **87 respondents** -- the same cohort members B and C use. Likert text
maps to 1-5; `"No Comments"` and blanks map to NaN, never to Neutral.
Per-item z-scoring before any similarity computation is mandatory here: the
raw responses have a ~80% Agree/Strongly-Agree ceiling that makes every pair
of students look near-identical.

In [1]:
import sys
from pathlib import Path

# This member's own modules. `common` needs no path work -- it is an
# installed package (pip install -e .), so it imports from any directory.
sys.path.insert(0, str(Path.cwd() / 'src'))

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

from load import load_clean, zscore_items, zscore_items_mean_imputed, SEED, N_RESPONDENTS_EXPECTED
from build import (
    pairwise_complete_cosine, mean_imputed_cosine,
    build_threshold_graph, build_mutual_knn_graph, build_disparity_backbone,
    percolation_sweep, graph_summary, edge_overlap,
    SIM_THRESHOLD, KNN_K, DISPARITY_ALPHA,
)
from metrics import (
    clustering_metrics, centralities, detect_communities, modularity,
    null_model_comparison, log_binned_degree_histogram,
    community_profile, community_assortativity, modularity_null,
)
from dynamics import (
    bounded_confidence_sweep, find_critical_epsilon, si_seeding_comparison,
    si_curve_comparison, seed_set_dispersion, SI_N_SEED_NODES,
)
from common import viz

viz.apply_style()

FIG_DIR = 'figures'
TAB_DIR = 'tables'

## 1. Load and z-score

In [2]:
coded = load_clean()
print(f'Respondents x items: {coded.shape} (expected {N_RESPONDENTS_EXPECTED} x 60)')
z = zscore_items(coded)

raw_corr = np.corrcoef(coded.fillna(coded.mean()).to_numpy())
iu = np.triu_indices(len(coded), 1)
print(f'Raw (non-zscored) pairwise correlation: mean={raw_corr[iu].mean():.3f}, min={raw_corr[iu].min():.3f}')

sim, shared = pairwise_complete_cosine(z)
svals = sim.to_numpy()[iu]
print(f'z-scored pairwise-complete cosine similarity: mean={svals.mean():.3f}, min={svals.min():.3f}, max={svals.max():.3f}')

fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(svals, bins=30, color=viz.CAT[0], edgecolor='white')
ax.set_xlabel('Pairwise cosine similarity (z-scored items)')
ax.set_ylabel('Count')
ax.set_title('Similarity distribution after per-item z-scoring')
fig.tight_layout()
fig.savefig(f'{FIG_DIR}/fig01_similarity_distribution.png')
plt.close(fig)

Respondents x items: (87, 60) (expected 87 x 60)
Raw (non-zscored) pairwise correlation: mean=0.267, min=-0.359
z-scored pairwise-complete cosine similarity: mean=-0.010, min=-0.638, max=0.758


## 2. Edge-rule comparison: threshold vs. mutual-kNN vs. disparity backbone

In [3]:
G_thresh = build_threshold_graph(sim, SIM_THRESHOLD)
G_knn = build_mutual_knn_graph(sim, KNN_K)
G_backbone = build_disparity_backbone(sim, DISPARITY_ALPHA, SIM_THRESHOLD)

summary_rows = []
for name, G in [('threshold', G_thresh), ('mutual_knn', G_knn), ('disparity_backbone', G_backbone)]:
    s = graph_summary(G)
    s['rule'] = name
    summary_rows.append(s)
edge_rule_df = pd.DataFrame(summary_rows)[['rule', 'n_nodes', 'n_edges', 'mean_degree', 'density', 'giant_component_fraction', 'n_isolates']]
edge_rule_df.to_csv(f'{TAB_DIR}/table01_edge_rule_comparison.csv', index=False)
print(edge_rule_df.to_string(index=False))

print(f"\nEdge overlap threshold vs kNN (Jaccard): {edge_overlap(G_thresh, G_knn):.3f}")
print(f"Edge overlap threshold vs backbone (Jaccard): {edge_overlap(G_thresh, G_backbone):.3f}")

              rule  n_nodes  n_edges  mean_degree  density  giant_component_fraction  n_isolates
         threshold       87      822    18.896552 0.219727                  1.000000           0
        mutual_knn       87      207     4.758621 0.055333                  1.000000           0
disparity_backbone       87        1     0.022989 0.000267                  0.022989          85

Edge overlap threshold vs kNN (Jaccard): 0.252
Edge overlap threshold vs backbone (Jaccard): 0.001


## 3. Percolation sweep -- giant component emergence

In [4]:
sweep = percolation_sweep(sim)
sweep.to_csv(f'{TAB_DIR}/table02_percolation_sweep.csv', index=False)

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(sweep['threshold'], sweep['giant_component_fraction'], marker='o', ms=3, color=viz.CAT[0])
ax.axvline(SIM_THRESHOLD, color=viz.INK_SOFT, linestyle='--', label=f'primary threshold={SIM_THRESHOLD}')
ax.set_xlabel('Similarity threshold')
ax.set_ylabel('Giant component fraction')
ax.set_title('Percolation: giant component vs. similarity threshold')
ax.legend()
fig.tight_layout()
fig.savefig(f'{FIG_DIR}/fig02_percolation_sweep.png')
plt.close(fig)

half = sweep[sweep['giant_component_fraction'] >= 0.5]
print(f"Giant component reaches >=50% of nodes at threshold >= {half.iloc[-1]['threshold'] if len(half) else 'never'}")

Giant component reaches >=50% of nodes at threshold >= 0.325


## 4. Degree distribution (log-binned), clustering, centralities

In [5]:
G = G_thresh  # primary graph for the rest of the analysis

hist = log_binned_degree_histogram(G)
hist.to_csv(f'{TAB_DIR}/table03_degree_histogram_logbinned.csv', index=False)

fig, ax = plt.subplots(figsize=(6, 4))
pos = hist[hist['bin_low'] > 0]
ax.loglog(pos['bin_low'], pos['density'], marker='o', ms=4, color=viz.CAT[0])
ax.set_xlabel('Degree (log)')
ax.set_ylabel('Density (log)')
ax.set_title('Log-binned degree distribution (threshold graph)')
fig.tight_layout()
fig.savefig(f'{FIG_DIR}/fig03_degree_distribution_logbinned.png')
plt.close(fig)

cm = clustering_metrics(G)
print('Clustering metrics:', cm)

cent = centralities(G)
cent.to_csv(f'{TAB_DIR}/table04_centralities.csv', index=False)
print('\nTop 5 eigenvector centrality:')
print(cent.head(5).to_string(index=False))
print('\nTop 5 betweenness centrality:')
print(cent.sort_values('betweenness', ascending=False).head(5).to_string(index=False))

Clustering metrics: {'global_clustering_transitivity': 0.5855077574047954, 'average_local_clustering': 0.5051391980294847, 'n_connected_components': 1, 'avg_shortest_path_giant_component': 2.054263565891473}

Top 5 eigenvector centrality:
node  degree  closeness  betweenness  eigenvector
 110      32   0.521212     0.009632     0.215642
 114      32   0.518072     0.009531     0.215051
  94      32   0.527607     0.013984     0.210978
  21      30   0.497110     0.005635     0.205764
  54      30   0.502924     0.006548     0.205460

Top 5 betweenness centrality:
node  degree  closeness  betweenness  eigenvector
  72      15   0.527607     0.042451     0.065476
  57      17   0.514970     0.036100     0.090750
  55      30   0.544304     0.031542     0.190400
  19      18   0.505882     0.030502     0.089614
  32      13   0.502924     0.026029     0.054180


## 5. Community detection and Erdos-Renyi null model comparison

In [6]:
comm_map, comms = detect_communities(G)
mod = modularity(G, comms)
print(f'Louvain communities: {len(comms)}, sizes: {sorted([len(c) for c in comms], reverse=True)}')
print(f'Modularity: {mod:.4f}')

comm_df = pd.DataFrame({'node': list(comm_map.keys()), 'community': list(comm_map.values())})
comm_df.to_csv(f'{TAB_DIR}/table05_communities.csv', index=False)

mod_null = modularity_null(G)
print(f"Modularity vs degree-preserving (configuration-model) null: "
      f"Q={mod_null['modularity_observed']:.4f} vs "
      f"{mod_null['modularity_null_mean']:.4f} +/- {mod_null['modularity_null_sd']:.4f} "
      f"(z = {mod_null['z_score']:+.1f})")
pd.DataFrame([mod_null]).to_csv(f'{TAB_DIR}/table11_modularity_null.csv', index=False)

null_cmp = null_model_comparison(G)
null_cmp.to_csv(f'{TAB_DIR}/table06_null_model_comparison.csv', index=False)
print('\nObserved vs ER null (n, mean-degree matched), mean +/- SD over 30 seeded realizations:')
print(null_cmp.to_string(index=False))

fig, ax = plt.subplots(figsize=(6, 4))
pos_layout = nx.spring_layout(G, seed=SEED)
# Explicit colours from the shared palette -- never a matplotlib colormap,
# which would auto-cycle and give each re-run a different look.
comm_ids = sorted(set(comm_map.values()))
comm_color = {cid: viz.CAT[k % len(viz.CAT)] for k, cid in enumerate(comm_ids)}
colors = [comm_color[comm_map[n]] for n in G.nodes()]
nx.draw_networkx_nodes(G, pos_layout, node_color=colors, node_size=60, ax=ax)
nx.draw_networkx_edges(G, pos_layout, alpha=0.15, ax=ax)
ax.set_title(f'Agreement network, colored by Louvain community (Q={mod:.3f})')
ax.axis('off')
fig.tight_layout()
fig.savefig(f'{FIG_DIR}/fig04_network_communities.png')
plt.close(fig)

Louvain communities: 4, sizes: [48, 35, 2, 2]
Modularity: 0.4320


Modularity vs degree-preserving (configuration-model) null: Q=0.4320 vs 0.1779 +/- 0.0072 (z = +35.5)



Observed vs ER null (n, mean-degree matched), mean +/- SD over 30 seeded realizations:
                           metric  observed  null_mean  null_sd
         average_local_clustering  0.505139   0.220126 0.009005
avg_shortest_path_giant_component  2.054264   1.792177 0.011470
         giant_component_fraction  1.000000   1.000000 0.000000


## 6. Bounded-confidence opinion dynamics (Deffuant-Weisbuch)

In [7]:
bc_sweep = bounded_confidence_sweep(G, n_trials=30)
bc_sweep.to_csv(f'{TAB_DIR}/table07_bounded_confidence_sweep.csv', index=False)
crit_eps = find_critical_epsilon(bc_sweep)
print(f'Critical epsilon (fragmentation -> consensus, >=50% trials): {crit_eps}')

fig, ax = plt.subplots(figsize=(6, 4))
ax.errorbar(bc_sweep['epsilon'], bc_sweep['mean_n_clusters'], yerr=bc_sweep['sd_n_clusters'],
            marker='o', ms=3, color=viz.CAT[0], ecolor=viz.AXIS_SOFT, capsize=2)
if crit_eps is not None:
    ax.axvline(crit_eps, color=viz.INK_SOFT, linestyle='--', label=f'critical epsilon={crit_eps}')
ax.set_xlabel('Confidence threshold (epsilon)')
ax.set_ylabel('Final number of opinion clusters (mean +/- SD, 30 trials)')
ax.set_title('Deffuant-Weisbuch: fragmentation -> consensus transition')
ax.legend()
fig.tight_layout()
fig.savefig(f'{FIG_DIR}/fig05_bounded_confidence_transition.png')
plt.close(fig)

Critical epsilon (fragmentation -> consensus, >=50% trials): 0.4


## 7. SI spreading: high-eigenvector-centrality vs. random seeding

In [8]:
hc_nodes = cent.sort_values('eigenvector', ascending=False)['node'].tolist()
si_cmp = si_seeding_comparison(G, hc_nodes, n_trials=30)
si_cmp.to_csv(f'{TAB_DIR}/table08_si_seeding_comparison.csv', index=False)
print(si_cmp.to_string(index=False))

fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(si_cmp['seeding'], si_cmp['mean_time_to_full_coverage'], yerr=si_cmp['sd_time_to_full_coverage'],
       color=[viz.CAT[0], viz.CAT[1]], capsize=4)
ax.set_ylabel('Mean time to full coverage (steps, 30 trials)')
ax.set_title('SI spreading: seeding strategy comparison')
fig.tight_layout()
fig.savefig(f'{FIG_DIR}/fig06_si_seeding_comparison.png')
plt.close(fig)

                    seeding  mean_final_coverage  sd_final_coverage  frac_trials_full_coverage  mean_time_to_full_coverage  sd_time_to_full_coverage
high_eigenvector_centrality             0.998467           0.004907                   0.900000                    9.111111                  1.065740
                     random             0.995019           0.007672                   0.666667                    7.750000                  0.942072


## 8. Community profiling -- what do the two blocs actually believe?

Louvain tells us *who* is grouped together; it does not say *why*. Here we
compare the two large communities item by item (mean z-score within each
bloc) to find the statements that actually separate them, and check how
segregated the blocs are with attribute assortativity.

In [9]:
prof = community_profile(z, comm_map)
prof.to_csv(f'{TAB_DIR}/table09_community_profile.csv', index=False)

sizes = pd.Series(comm_map).value_counts()
big = [c for c in sizes.index if sizes[c] >= 5]
c0, c1 = big[0], big[1]
colA, colB = f'comm{c0}_mean_z', f'comm{c1}_mean_z'
print(f'Large communities: comm{c0} (n={sizes[c0]}), comm{c1} (n={sizes[c1]})')
print(f'Overall mean z: comm{c0}={prof[colA].mean():+.3f}, comm{c1}={prof[colB].mean():+.3f}')

r_comm = community_assortativity(G, comm_map)
print(f'Community attribute assortativity: {r_comm:+.3f}')

# c0 is the larger (moderate) bloc; report the gap from the high-conviction side
prof['gap'] = prof[colB] - prof[colA]
prof.to_csv(f'{TAB_DIR}/table09_community_profile.csv', index=False)
top = prof.sort_values('abs_gap', ascending=False).head(20).iloc[::-1]
print('\nBlock composition of the 20 most discriminating items:',
      top['block'].value_counts().to_dict())
print('\nTop 8 discriminating items:')
print(prof.sort_values('abs_gap', ascending=False)
        .head(8)[['item', 'block', colA, colB, 'gap']].to_string(index=False))

block_gap = prof.groupby('block')['abs_gap'].mean().reindex(['T', 'E', 'S', 'V'])

fig, axes = plt.subplots(1, 2, figsize=(11, 5.2), gridspec_kw={'width_ratios': [2.1, 1]})

ax = axes[0]
ax.barh(top['item'], top['gap'],
        color=[viz.BLOCK_COLORS[b] for b in top['block']], height=0.72)
ax.axvline(0, color=viz.INK_SOFT, lw=1)
ax.set_xlabel(f'Mean z-score gap  (high-conviction bloc n={sizes[c1]}  minus  moderate bloc n={sizes[c0]})')
ax.set_title('The 20 statements that separate the two blocs')
ax.grid(axis='y', visible=False)
handles = [plt.Rectangle((0, 0), 1, 1, color=viz.BLOCK_COLORS[b]) for b in 'TESV']
ax.legend(handles, [viz.BLOCK_NAMES[b] for b in 'TESV'], loc='upper center',
          bbox_to_anchor=(0.5, -0.12), ncol=4, fontsize=8.5)

ax = axes[1]
ax.bar(block_gap.index, block_gap.values,
       color=[viz.BLOCK_COLORS[b] for b in block_gap.index], width=0.65)
for b, v in block_gap.items():
    ax.text(b, v, f'{v:.2f}', ha='center', va='bottom', fontsize=9, color=viz.INK)
ax.set_ylabel('Mean |z-gap| per item')
ax.set_title('Which topics divide the class')
ax.grid(axis='x', visible=False)

fig.suptitle('Community profiling: the split is in intensity, and it lives in S and V',
             fontsize=12, fontweight='bold', y=1.0)
fig.tight_layout()
fig.savefig(f'{FIG_DIR}/fig07_community_profile.png')
plt.close(fig)

Large communities: comm0 (n=48), comm1 (n=35)
Overall mean z: comm0=-0.283, comm1=+0.371
Community attribute assortativity: +0.821

Block composition of the 20 most discriminating items: {'V': 11, 'S': 7, 'E': 1, 'T': 1}

Top 8 discriminating items:
item block  comm0_mean_z  comm1_mean_z      gap
 S09     S     -0.543483      0.757321 1.300804
 V10     V     -0.495934      0.689703 1.185637
 S14     S     -0.524818      0.643360 1.168177
 V07     V     -0.535216      0.618342 1.153558
 V08     V     -0.490346      0.612347 1.102693
 V12     V     -0.461148      0.607126 1.068273
 V15     V     -0.461281      0.561645 1.022926
 V11     V     -0.496008      0.525792 1.021800


## 9. Does seed choice matter on a sparser graph?

Section 7 found no advantage for high-centrality seeding on the threshold
graph. The obvious suspicion is that the graph is simply too dense (mean
degree ~19 of 86 possible) for seed placement to matter. The mutual-kNN graph
uses the same students and the same similarity data but has mean degree ~4.8,
so it is the natural control.

On the sparse graph the epidemic never reaches everyone, so time-to-full-
coverage is undefined; we compare the full epidemic curves (coverage at each
step) instead, which is well defined on both graphs.

In [10]:
cent_knn = centralities(G_knn)
hc_knn = cent_knn.sort_values('eigenvector', ascending=False)['node'].tolist()

curves = pd.concat([
    si_curve_comparison(G, cent.sort_values('eigenvector', ascending=False)['node'].tolist()).assign(graph='threshold'),
    si_curve_comparison(G_knn, hc_knn).assign(graph='mutual_knn'),
])
curves.to_csv(f'{TAB_DIR}/table10_si_coverage_curves.csv', index=False)

final = curves[curves['step'] == curves['step'].max()]
print('Coverage at t=30:')
print(final[['graph', 'seeding', 'mean_coverage', 'sd_coverage']].to_string(index=False))

# Why does centrality seeding underperform on the sparse graph? Check whether
# the top-centrality seeds are simply sitting on top of each other.
for name, Gx, ranking in [('threshold', G, cent.sort_values('eigenvector', ascending=False)['node'].tolist()),
                          ('mutual_knn', G_knn, hc_knn)]:
    hc = ranking[:SI_N_SEED_NODES]
    rnd = list(np.random.default_rng(SEED).choice(list(Gx.nodes()), size=SI_N_SEED_NODES, replace=False))
    print(f'{name}: seed dispersion  high-centrality={seed_set_dispersion(Gx, hc):.2f} hops, '
          f'random={seed_set_dispersion(Gx, rnd):.2f} hops')

fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.2), sharey=True)
titles = {'threshold': f'Threshold graph (mean degree {2 * G.number_of_edges() / G.number_of_nodes():.1f})',
          'mutual_knn': f'Mutual-kNN graph (mean degree {2 * G_knn.number_of_edges() / G_knn.number_of_nodes():.1f})'}
styles = {'high_eigenvector_centrality': (viz.CAT[0], 'high-eigenvector seeds'),
          'random': (viz.CAT[1], 'random seeds')}

for ax, gname in zip(axes, ['threshold', 'mutual_knn']):
    endpoints = {}
    for seeding, (colour, label) in styles.items():
        sub = curves[(curves['graph'] == gname) & (curves['seeding'] == seeding)]
        ax.plot(sub['step'], sub['mean_coverage'], color=colour, lw=2, label=label)
        lo = (sub['mean_coverage'] - sub['sd_coverage']).clip(0, 1)
        hi = (sub['mean_coverage'] + sub['sd_coverage']).clip(0, 1)
        ax.fill_between(sub['step'], lo, hi, color=colour, alpha=0.15, lw=0)
        endpoints[seeding] = sub['mean_coverage'].iloc[-1]
    # label the endpoints only where the two curves actually separate
    ys = list(endpoints.values())
    if abs(ys[0] - ys[1]) > 0.03:
        for seeding, y in endpoints.items():
            ax.annotate(f'{y:.2f}', (curves['step'].max(), y), xytext=(5, 0),
                        textcoords='offset points', va='center', fontsize=9,
                        color=viz.INK)
    else:
        ax.annotate(f'both {ys[0]:.2f}', (curves['step'].max(), ys[0]), xytext=(5, 0),
                    textcoords='offset points', va='center', fontsize=9, color=viz.INK)
    ax.set_xlabel('SI time step')
    ax.set_title(titles[gname])
    ax.set_xlim(0, 34)
    ax.set_ylim(-0.02, 1.05)
    ax.grid(axis='x', visible=False)

axes[0].set_ylabel('Fraction of class reached (mean of 30 trials)')
axes[0].legend(loc='lower right')
fig.suptitle('Seeding the best-connected students backfires once the network is sparse',
             fontsize=12, fontweight='bold')
fig.tight_layout()
fig.savefig(f'{FIG_DIR}/fig08_si_seeding_density.png')
plt.close(fig)

Coverage at t=30:
     graph                     seeding  mean_coverage  sd_coverage
 threshold high_eigenvector_centrality       0.998467     0.004907
 threshold                      random       0.995019     0.007672
mutual_knn high_eigenvector_centrality       0.244061     0.265891
mutual_knn                      random       0.486590     0.400679
threshold: seed dispersion  high-centrality=1.00 hops, random=2.00 hops
mutual_knn: seed dispersion  high-centrality=1.00 hops, random=3.33 hops


## 10. Export for Gephi

Both graphs are written out as `.gexf` with the Louvain community and the
centrality scores attached as node attributes, so the layouts in the report
can be reproduced interactively.

In [11]:
import os
os.makedirs('gexf', exist_ok=True)

cent_idx = cent.set_index('node')
for name, Gx in [('agreement_threshold', G), ('agreement_knn', G_knn)]:
    H = Gx.copy()
    nx.set_node_attributes(H, comm_map, 'community')
    for col in ['degree', 'betweenness', 'eigenvector']:
        nx.set_node_attributes(H, cent_idx[col].to_dict(), col)
    nx.write_gexf(H, f'gexf/{name}.gexf')
    print(f'gexf/{name}.gexf  ({H.number_of_nodes()} nodes, {H.number_of_edges()} edges)')

gexf/agreement_threshold.gexf  (87 nodes, 822 edges)
gexf/agreement_knn.gexf  (87 nodes, 207 edges)
